# WGS pipelineOrchestrates the genotype side end to end: acquisition, per-callset filter/QC, normalization,merge, relatedness, dedup, ancestry QC/PCA, GWAS. The clinical side is `clinical_core.py`(before step 1) and `analysis_grain.py` (after step 6), both cluster-only.Runs locally; cells that touch the cluster do so over `ssh`/`sbatch`. Run order and rationale:`README.md` and `METHODS.md`.## 0. Data acquisition — provenanceWhere every input came from, genotype and clinical alike. Kept here rather than run once anddiscarded, so the pull is part of the record and re-runnable — `synapse get` resumes.

In [ ]:
import subprocess

# The ONE cluster path in this notebook. Everything remote is derived from it.
# It is the project root on biowulf: code and data share a root there, so this is
# simultaneously $WGS_ROOT, $PROJECT_ROOT and the old $BUNDLE — they are one thing now.
REMOTE_ROOT = "/data/CARDPB2/sysbio/wgs"

HELIX = "helix.nih.gov"   # transfer node — Synapse/large downloads go here, never through biowulf


def run_on_helix(cmd):
    """Run a shell command on helix.nih.gov over ssh, streaming output, raising on failure."""
    subprocess.run(["ssh", HELIX, cmd], check=True)

### DivCo_HS

In [ ]:
DIVCO_DEST = f"{REMOTE_ROOT}/data/amp-ad-genomics/DivCo_HS/joint_calls"
DIVCO_MERGED = "syn68260951"        # merged.deduped.vcf.gz
DIVCO_MERGED_TBI = "syn68260952"    # merged.deduped.vcf.gz.tbi

run_on_helix(f"mkdir -p {DIVCO_DEST} && "
             f"synapse get {DIVCO_MERGED} --downloadLocation {DIVCO_DEST} && "
             f"synapse get {DIVCO_MERGED_TBI} --downloadLocation {DIVCO_DEST}")

### WGS_Harmonization

In [ ]:
WGS_HARM_DEST = f"{REMOTE_ROOT}/data/amp-ad-genomics/WGS_Harmonization/joint_calls"
WGS_HARM_FOLDER = "syn11707420"    # 24 chromosome VCFs (1-22, X, Y)

run_on_helix(f"mkdir -p {WGS_HARM_DEST} && "
             f"synapse get -r {WGS_HARM_FOLDER} --downloadLocation {WGS_HARM_DEST}")

### WB-DWGS

In [ ]:
WB_DWGS_DEST = f"{REMOTE_ROOT}/data/amp-pd-genomics/WB-DWGS/joint_calls"
WB_DWGS_SRC = "gs://amp-pd-genomics/releases/2023_v4release_1027/wgs-WB-DWGS/plink/pfiles/all_chrs_merged.{pgen,psam,pvar,log}"

run_on_helix(f"mkdir -p {WB_DWGS_DEST} && "
             f"gcloud storage cp {WB_DWGS_SRC} {WB_DWGS_DEST}/ --billing-project 8641313829")

### BR-DSNWGS

In [ ]:
BR_DSNWGS_DEST = f"{REMOTE_ROOT}/data/amp-pd-genomics/BR-DSNWGS/joint_calls"
BR_DSNWGS_SRC = "gs://amp-pd-receipt-2026/mssm/20260626-transfer-mssm-jvcf/*"

run_on_helix(f"mkdir -p {BR_DSNWGS_DEST} && "
             f'gcloud storage cp "{BR_DSNWGS_SRC}" {BR_DSNWGS_DEST}/ --billing-project 8641313829')

### Clinical metadataThe eleven files `clinical_core.py` opens. Moved here from that script's header 2026-08-21, wherethey sat as unrunnable comments — §0 is the single acquisition record.**Known gap:** the cluster's `DivCo_HS/metadata` holds 5 files and only these 3 syn IDs arerecorded. The other two have no recorded provenance (HANDOFF known issue 4).

In [ ]:
AD_META    = f"{REMOTE_ROOT}/data/amp-ad-genomics/WGS_Harmonization/metadata"DIVCO_META = f"{REMOTE_ROOT}/data/amp-ad-genomics/DivCo_HS/metadata"PD_META    = f"{REMOTE_ROOT}/data/amp-pd-genomics/metadata"DIVCO_SYN = ["syn51757644", "syn51757645", "syn51757646"]AD_SYN    = ["syn73713768", "syn12178037", "syn73713767", "syn21893059", "syn73713766"]run_on_helix(f"mkdir -p {DIVCO_META} && "             + " && ".join(f"synapse get {s} --downloadLocation {DIVCO_META}" for s in DIVCO_SYN))run_on_helix(f"mkdir -p {AD_META} && "             + " && ".join(f"synapse get {s} --downloadLocation {AD_META}" for s in AD_SYN))

In [ ]:
PD_RELEASE = "gs://amp-pd-data/releases/2023_v4release_1027"PD_FILES = ["clinical/Demographics.csv", "amp_pd_participants.csv",            "amp_pd_case_control.csv", "wgs_BR-DSNWGS_sample_inventory.csv"]run_on_helix(f"mkdir -p {PD_META} && "             + " && ".join(f"gcloud storage cp {PD_RELEASE}/{f} {PD_META}/ "                           f"--billing-project 8641313829" for f in PD_FILES))

## 1. VCF → pgen

Only BR-DSNWGS needs this. How each callset got to pgen:

| Callset | Route |
|---|---|
| `wgs_harm` | 24 per-chromosome **b37** VCFs → per-chrom `plink2` filter → `--pmerge-list` → `liftOver` b37→hg38 on the *merged* file → `--chr 1-22,X,Y` |
| `divco_hs` | one pre-merged hg38 VCF → `bcftools` filter → `plink2 --make-pgen` |
| `wb_dwgs` | none — AMP-PD ships plink2 pfiles in hg38 |
| `br_dsnwgs` | this section — same two stages as `divco_hs` |

**chrX needs a sex file.** plink2 refuses to import non-PAR X without per-sample sex, so
stage 2 writes an all-unknown placeholder purely to let the import proceed. Real sex comes
from `clinical_core` §10 and is applied by genetics step 1, where the genotype sex-check
also adjudicates it.

Chromosome naming and variant IDs need **not** match the other callsets —
`02_normalize.sh` re-runs `--output-chr chrM --set-all-var-ids '@:#:$r:$a'` across all of
them. This section only has to produce a valid pgen.

In [ ]:
BIOWULF = "biowulf.nih.gov"   # compute; bulk transfers go to helixWGS_ROOT = REMOTE_ROOTBUNDLE = REMOTE_ROOT   # code lives AT the root now, not in a sub-bundleLOGS = f"{BUNDLE}/logs"def submit_step(script, name, sbatch_args=(), **exports):    """Submit a bundle step through submit.sh, which resolves $BUNDLE and sources config.sh.    sbatch_args overrides the script's own #SBATCH directives (sbatch takes the command line    over the file). Pass resources here rather than typing them at a shell, so the notebook    stays a faithful record of what was actually submitted — submit.sh logs them either way,    but only this way can the run be reproduced from the notebook.    """    ex = ",".join(f"{k}={v}" for k, v in exports.items())    extra = (" " + " ".join(sbatch_args)) if sbatch_args else ""    r = subprocess.run(["ssh", BIOWULF, f"cd {BUNDLE} && ./submit.sh {script} "                                        f"--job-name={name}{extra} --export={ex} --parsable"],                       capture_output=True, text=True)    if r.returncode:        raise RuntimeError(r.stderr.strip() or r.stdout.strip())    jid = next((t for t in reversed(r.stdout.split()) if t.isdigit()), "?")    print(f"job {jid}   tail -f {LOGS}/{name}.o")    return jiddef sbatch_on_biowulf(name, body, time="12:00:00", cpus=16, mem="64g"):    """Write <name>.sh on biowulf from `body`, submit it, return the job id.    For one-off work that has no script in the bundle — currently only §1."""    script = f"{BUNDLE}/generated/{name}.sh"    header = (f"#!/bin/bash\n#SBATCH --job-name={name}\n"              f"#SBATCH --output={LOGS}/{name}.o\n#SBATCH --error={LOGS}/{name}.e\n"              f"#SBATCH --time={time}\n#SBATCH --cpus-per-task={cpus}\n"              f"#SBATCH --mem={mem}\n#SBATCH --partition=norm\nset -euo pipefail\n\n")    subprocess.run(["ssh", BIOWULF, f"mkdir -p $(dirname {script}) {LOGS} && cat > {script}"],                   input=header + body, text=True, check=True)    jid = subprocess.run(["ssh", BIOWULF, f"sbatch --parsable {script}"],                         capture_output=True, text=True, check=True).stdout.strip()    print(f"job {jid}   tail -f {LOGS}/{name}.o")    return jiddef run_on_biowulf(cmd, check=True):    """Run a shell command on biowulf over ssh, capturing output. Compute node, not transfers."""    r = subprocess.run(["ssh", BIOWULF, cmd], capture_output=True, text=True)    print(r.stdout.strip() or "(no stdout)")    if r.stderr.strip():        print("stderr:", r.stderr.strip())    if check and r.returncode:        raise RuntimeError(f"exit {r.returncode}")    return r.stdout

In [ ]:
BR = f"{WGS_ROOT}/data/amp-pd-genomics/BR-DSNWGS"
BR_VCF = f"{BR}/joint_calls/AMPPD_postmortem_joint_gt_call_97donors.vcf.gz"
BR_FILTERED = f"{BR}/pgen/intermediate/br_dsnwgs_filtered.vcf.gz"
BR_PGEN = f"{BR}/pgen/br_dsnwgs_hg38"

BUILD = "hg38"   # b37 would need a liftover stage instead — see wgs_harm_stage3_liftover.sh

### Stage 1 — filter

Biallelic SNPs, IDs set to `CHROM:POS:REF:ALT`. `--apply-filters 'PASS,.'` keeps passing
and unfiltered sites and drops explicit VQSR failures, so it is correct whether or not this
callset was recalibrated. The job echoes the depositor README and the `##reference` header
first, so the build it actually ran on is on the record.

In [ ]:
s1 = sbatch_on_biowulf("br_dsnwgs_s1_filter", f"""
module load bcftools
mkdir -p $(dirname {BR_FILTERED})

cat {BR}/joint_calls/README.txt || true
bcftools view -h {BR_VCF} | grep -i '^##reference' || echo "no ##reference header"
echo "samples in: $(bcftools query -l {BR_VCF} | wc -l)"

bcftools view \\
    --apply-filters 'PASS,.' \\
    --min-alleles 2 --max-alleles 2 --type snps \\
    {BR_VCF} \\
  | bcftools annotate --set-id '%CHROM:%POS:%REF:%ALT' \\
  | bgzip -@ 8 > {BR_FILTERED}

tabix -p vcf {BR_FILTERED}
echo "variants out: $(bcftools index -n {BR_FILTERED})"
""")

### Stage 2 — VCF → pgen

`--chr 1-22,X,Y` drops alt and unplaced contigs and accepts either `chr1` or `1`.
`--split-par` separates the chrX pseudo-autosomal regions.

In [ ]:
s2 = sbatch_on_biowulf("br_dsnwgs_s2_vcf_to_pgen", f"""
module load bcftools plink/6-alpha
mkdir -p {BR}/pgen

SEX=$(dirname {BR_FILTERED})/placeholder_sex.txt
bcftools query -l {BR_FILTERED} | awk '{{print $1, $1, 0}}' > "$SEX"

plink2 \\
    --vcf {BR_FILTERED} \\
    --chr 1-22,X,Y \\
    --split-par {BUILD} \\
    --update-sex "$SEX" \\
    --make-pgen \\
    --threads 16 \\
    --out {BR_PGEN}

echo "variants: $(grep -vc '^#' {BR_PGEN}.pvar)"
echo "samples:  $(grep -vc '^#' {BR_PGEN}.psam)"
""", time="24:00:00", mem="128g")

## 2. GenoTools — per-callset filter, ancestry, QC

`scripts/01_genotools.sh` runs once per callset: applies the sex file, filters to biallelic
PASS SNPs, converts to bed, then projects onto the reference panel for ancestry and runs QC.
It writes the two things later steps read back — the `<ANC>_pass_fail` JSON (step 5 pulls
QC-fail reasons from it) and the predicted ancestry labels (steps 4 and 6).

From here on every step is a script in `scripts/`, submitted through `submit.sh`. The
notebook supplies the run order, the per-step variables, and the pulls in between; the bash
stays in the bundle so the notebook and the cluster can't drift apart.

Only `br_dsnwgs` needs a run — the other three came through the validation pass.

### Push the code — `git pull` on the clusterThe repo has a remote as of 2026-08-21, so code reaches biowulf by `git pull` and never by`rsync`. This retires the project's most expensive recurring trap: `rsync` without `--delete`cannot express a deletion, which caused three separate stale-artifact bugs. Git can.The pull prints the commit, so every job below is traceable to a SHA.

In [ ]:
run_on_biowulf(f"cd {REMOTE_ROOT} && git pull --ff-only && git log -1 --oneline")

In [ ]:
DATA = f"{WGS_ROOT}/data"   # same layout as config.sh and clinical_core.py

# callset -> (base dir, pgen prefix relative to it). Mirrors config.sh's DIR_*/RAW_*.
# Note the asymmetry: WB-DWGS keeps its pgen in joint_calls/, the rest in pgen/.
CALLSETS = {
    "wgs_harm":  (f"{DATA}/amp-ad-genomics/WGS_Harmonization", "pgen/wgs_harm_hg38"),
    "divco_hs":  (f"{DATA}/amp-ad-genomics/DivCo_HS",          "pgen/divco_hs_hg38"),
    "wb_dwgs":   (f"{DATA}/amp-pd-genomics/WB-DWGS",           "joint_calls/all_chrs_merged"),
    "br_dsnwgs": (f"{DATA}/amp-pd-genomics/BR-DSNWGS",         "pgen/br_dsnwgs_hg38"),
}


def genotools(name, cpus=16, mem="128g", time="12:00:00"):
    """Step 1 for one callset.

    cpus is deliberately modest. The Aug-7 br_dsnwgs run died in step 4 with 192 threads on a
    64-CPU allocation: plink2 honours --threads, but genotools' Python stack (OpenBLAS,
    sklearn, numba, xgboost) sizes itself off the NODE core count. 01_genotools.sh now caps
    every pool at $SLURM_CPUS_PER_TASK, so cpus here sets the real thread budget — and since
    each thread carries its own workspace, it sets the memory ceiling too. More is not better.
    """
    base, pgen = CALLSETS[name]
    return submit_step("scripts/01_genotools.sh", f"genotools_{name}",
                       sbatch_args=(f"--cpus-per-task={cpus}", f"--mem={mem}", f"--time={time}"),
                       PGEN=f"{base}/{pgen}",
                       SEX_FILE=f"{WGS_ROOT}/clinical_core_out/{name}_update_sex.txt",
                       OUT_DIR=f"{base}/genotools",
                       DATASET=name)

In [ ]:
genotools("br_dsnwgs")

## 3. Ancestry QC + PCA — step 6, in one submission

Step 6 does per-ancestry variant QC, builds the AF-concordance exclusion list **from that
unfiltered output**, applies it, and runs prune+PCA on *both* generations — so the filter's
effect is measured on every run rather than asserted from a number someone typed into a log.

It used to be two submissions of step 6 with `af_concordance_build.sh` and a manual `mv`
between them. That shape was justified by a circular ordering — `grain <- §12 <- PCs <- step 6` —
which turned out not to exist. Two facts dissolve it, both written up in the header of
`scripts/06_ancestry_qc.sh`:

1. the AF build never needed the grain, only `IID -> (source_callset, dx_detailed)`, which is
   pure clinical output and is now written separately as `sample_annot.csv` by §12a;
2. `--geno`/`--maf`/`--hwe` are per-variant on a fixed sample set, so they **commute** with
   `--exclude` — pass 2 was re-scanning 527 GiB to recompute numbers it already had.

**Prereq:** `clinical_core.py` has been run on the cluster. §12a writes `sample_annot.csv`,
and it needs no PCs, so there is no chicken-and-egg left to break. (§11-13 now live in
`analysis_grain.py` and run *after* this step — see the Next cell.)

### One-time migration — adopt the existing unfiltered pass as stage A

Stage A's inputs are `cohort_merged`, the step-5 manifest and the locked thresholds. None of
those changed, so the pass-1 output already sitting in `by_ancestry_qc/` **is** stage A's
output — rebuilding it would reproduce the same bytes at the cost of another full scan of the
merged bed, ~13-14 min x 11 strata.

Moving it into `unfiltered/` lets step 6's reuse rule (skip stage A when a valid fileset
postdates the merge) pick it up. Idempotent: re-running this prints `already migrated`.

In [ ]:
run_on_biowulf(f"""
set -e
cd {BUNDLE} && source config.sh
D="${{MERGED_DIR}}/by_ancestry_qc"
if [ -d "$D/unfiltered" ]; then
    echo "already migrated — $(ls "$D/unfiltered"/cohort_*_qc.bed 2>/dev/null | wc -l) filesets"
    exit 0
fi
mkdir -p "$D/unfiltered"
mv "$D"/cohort_*_qc.* "$D"/cohort_*_pca.* "$D"/*_prune.* "$D/unfiltered"/ 2>/dev/null || true
echo "migrated $(ls "$D/unfiltered"/cohort_*_qc.bed | wc -l) filesets into $D/unfiltered"
""")

### Submit

One job. Stage A is reused (above), so what actually runs is the AF build, the exclusion, and
the filtered prune+PCA — strictly less work than the old pass 2 alone.

**Stage B doubles as the regression test for the split.** It rebuilds the exclusion list
through `sample_annot.csv` instead of `analysis_grain.csv`. Job 27697096 produced **4,415**
variants from the grain; the same count here means the refactor changed nothing. A different
count means the two reconciliation paths disagree — stop and read §12a's self-check before
trusting the output.

Knobs: `AF_EXCLUDE=none` runs deliberately unfiltered - `SKIP_AF_BUILD=1` reuses the list on
disk - `FORCE_QC=1` rebuilds stage A from scratch.

In [ ]:
s6 = submit_step("scripts/06_ancestry_qc.sh", "ancestry_qc")

## 4. Before and after — does the filter actually work?

The proof figure. Both manifests come out of the **same** step-6 job, which is what makes the
delta trustworthy: an earlier version of this comparison was assembled from two separate runs,
and one of them had silently picked up a stale exclusion list, so the "unfiltered" baseline
was not unfiltered.

Read it as three claims:

| panel | claim |
|---|---|
| dumbbell | how much of each PC was callset, and how much of that the filter removed |
| EUR scatters | the mechanism — a concentrated set of variants carried the separation |
| AJ scatters | **where it does not work.** AJ's split survives the filter |

The AJ row is in the figure on purpose. No AJ cell was ever powered enough to evaluate
(`AJ/control` is `wgs_harm`=44, under `MIN_CELL`), so the list holds no AJ-derived flags and
EUR-derived flags do not transfer. The leading hypothesis is that AJ's split is real
sub-continental structure rather than artifact — untested; see `PROJECT_LOG.md`.

In [ ]:
from pathlib import Path
from IPython.display import Image, display

LOCAL_PCA = Path("results/pca")
LOCAL_PCA.mkdir(parents=True, exist_ok=True)
REMOTE_QC = f"{REMOTE_ROOT}/data/merged/by_ancestry_qc"

# The two generations. Same job, same samples, same pruning and PCA settings — the ONLY
# difference is the exclusion list, which is the whole basis of the comparison.
for gen, sub in (("before", "unfiltered/"), ("after", "")):
    subprocess.run(["rsync", "-av",
                    f"{HELIX}:{REMOTE_QC}/{sub}retained_samples_manifest.csv",
                    str(LOCAL_PCA / f"manifest_{gen}.csv")], check=True)

subprocess.run(["python3", "review/plot_af_filter_effect.py",
                "--before", str(LOCAL_PCA / "manifest_before.csv"),
                "--after",  str(LOCAL_PCA / "manifest_after.csv")], check=True)

display(Image(str(LOCAL_PCA / "af_filter_effect.png")))

### Next

PCs have changed, so the grain must be rebuilt on them before the GWAS reads it. That is
`analysis_grain.py` — §11-13, split out of `clinical_core.py` so the second half of the clinical
side reads §9's audit tables instead of re-deriving them (and so it stops rewriting the sex files
step 1 already consumed):

```
module load python/3.11 && source .venv/bin/activate
python3 analysis_grain.py                # §12 rebuilds analysis_grain.csv on the new PCs
./submit.sh scripts/07_gwas.sh
```

Both clinical scripts run on the **cluster only** — the two machines hold different clinical
inputs. See `HANDOFF.md` known issue 4.